In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
# import seaborn as sns
# import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(216, 11)


,id,template,placeholders,session,cam,constructed_filename,label,top1_pred,correct_top1,top5_preds,correct_top5
123,260930-142122,Putting [something] onto [something],"magnet, electrical extension",magnet_2,cam0,260930-142122_cam0.avi,112,49,False,"[106, 109, 100, 1, 49]",False
130,260930-142212,Pushing [something] with [something],"magnet, wallet",magnet_2,cam1,260930-142212_cam1.avi,101,101,True,"[94, 15, 42, 157, 101]",True


In [5]:
# rename top1_label to top1_pred
result = result.rename(columns={'id':'video_id'})

del result['constructed_filename']

result.sample(3)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
174,260930-143737,Pushing [something] with [something],"wallet, books",books_2,cam0,101,101,True,"[18, 100, 42, 118, 101]",True
137,260930-143319,Putting [something] next to [something],"books, phone",books_2,cam2,107,105,False,"[6, 107, 114, 118, 105]",True
100,260930-141846,Putting [something] on a surface,magnet,magnet_2,cam1,109,109,True,"[155, 119, 116, 108, 109]",True


In [6]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(216, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [7]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 216
Top-1 accuracy: 0.4491
Top-5 accuracy: 0.7593


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [8]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.42
cam1,0.51
cam2,0.42


In [9]:
result.groupby(['cam']).agg(
    # count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,top1_acc,top5_acc
cam,,
cam0,0.42,0.74
cam1,0.51,0.86
cam2,0.42,0.68


In [10]:
result.groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],9,0.22,0.44
Moving [something] up,36,0.39,0.75
Piling [something] up,9,0.22,0.67
Plugging [something] into [something],9,0.89,0.89
Pushing [something] with [something],36,0.69,0.97
Putting [something] next to [something],36,0.42,0.78
Putting [something] on a surface,36,0.56,0.81
Putting [something] onto [something],36,0.19,0.61
Showing a photo of [something] to the camera,9,0.44,0.56


In [11]:
result[result.cam=='cam1'].groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],3,0.33,0.67
Moving [something] up,12,0.50,1.00
Piling [something] up,3,0.00,0.67
Plugging [something] into [something],3,1.00,1.00
Pushing [something] with [something],12,0.75,1.00
Putting [something] next to [something],12,0.75,1.00
Putting [something] on a surface,12,0.50,0.75
Putting [something] onto [something],12,0.08,0.58
Showing a photo of [something] to the camera,3,0.67,1.00


In [12]:
result[result.cam=='cam1'].groupby(['session']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
session,,,
attaching_1,3,0.33,0.67
books_2,15,0.33,0.73
lady_2,15,0.60,0.93
magnet_2,15,0.67,1.00
piling_1,3,0.00,0.67
plug_2,15,0.47,0.80
plugging_1,3,1.00,1.00
showing_photo_of_1,3,0.67,1.00


# Granular analysis

##### id2label

In [13]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [14]:
result.head(1)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
0,260930-122254,Putting [something] next to [something],"electrical extension, plug",plug_2,cam0,107,107,True,"[49, 19, 100, 73, 107]",True


In [15]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(216, 9)


,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True
1,260930-122254,cam1,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True


In [16]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260930-122254,cam0,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True,0.000008,...,0.001373,0.000001,0.000002,0.000003,0.000000,0.000000,0.000044,0.000010,0.000014,0.000002
1,260930-122254,cam1,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True,0.000000,...,0.000002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000005,0.000000,0.000000


#### Main Analysis 


In [17]:
pd.set_option('display.float_format',  '{:,.4f}'.format)

In [18]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

### "special objects in common actions":

- plug_2 shows 15 videos of common actions being performed with object plug. We should see a spike in 'Plugging [something] into [something]' for these. 
- So plug_2 is actually where plug is a distractor.

- I have plug_2, lady_2, magnet_2, books_2 for "special objects in common actions" in recording session 2

In [19]:
df.session.unique()

array(['plug_2', 'lady_2', 'magnet_2', 'books_2', 'plugging_1',
       'showing_photo_of_1', 'attaching_1', 'piling_1'], dtype=object)

In [21]:
avg_prob_in_diff_cases(df, session_type='plug_2', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug_2 =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  plug_2      15     0.0415    0.0721   0.4667  260930-122254, 260930-122317, 260930-122340
==================== Videos ====================
        video_id   cam session                placeholders                                 template label top1_pred correct_top1 correct_top5 prob_107 prob_73 prob_11 prob_19 prob_42
1  260930-122254  cam1  plug_2  electrical extension, plug  Putting [something] next to [something]   107       107         True         True   0.9830  0.0062  0.0052  0.0018  0.0009

Top 5 Probability Breakdown:
  prob_107 ->  0.98298  |  Putting [something] next to [something]
  prob_73  ->  0.00617  |  Pretending to put [something] next to [something]
  prob_11  ->  0.00520  |  Dropping [something] next to [something]
  prob_19  ->  0.00181  |  Hol

In [22]:
avg_prob_in_diff_cases(df, session_type='lady_2', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady_2 =========================
label 124: Showing a photo of [something] to the camera
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  lady_2      15     0.0003    0.0006   0.6000  260930-124820, 260930-124846, 260930-124905
==================== Videos ====================
         video_id   cam session           placeholders                                 template label top1_pred correct_top1 correct_top5 prob_107 prob_11 prob_42 prob_19 prob_101
46  260930-124820  cam1  lady_2  photo of lady, wallet  Putting [something] next to [something]   107       107         True         True   0.6515  0.1736  0.1461  0.0087   0.0051

Top 5 Probability Breakdown:
  prob_107 ->  0.65152  |  Putting [something] next to [something]
  prob_11  ->  0.17360  |  Dropping [something] next to [something]
  prob_42  ->  0.14609  |  Moving [something] closer to [something]
  prob_19  ->  0.00866  |  Holding [s

In [23]:
avg_prob_in_diff_cases(df, session_type='books_2', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books_2 =========================
label 48: Piling [something] up
cams used: ['cam1']
   session  videos  prob_mean  prob_std  top1acc                                     examples
0  books_2      15     0.1313    0.1979   0.3333  260930-143319, 260930-143358, 260930-143410
==================== Videos ====================
          video_id   cam  session  placeholders                                 template label top1_pred correct_top1 correct_top5 prob_11 prob_107 prob_126 prob_133 prob_19
136  260930-143319  cam1  books_2  books, phone  Putting [something] next to [something]   107        11        False         True  0.5232   0.1924   0.1322   0.0442  0.0338

Top 5 Probability Breakdown:
  prob_11  ->  0.52322  |  Dropping [something] next to [something]
  prob_107 ->  0.19243  |  Putting [something] next to [something]
  prob_126 ->  0.13224  |  Showing [something] next to [something]
  prob_133 ->  0.04424  |  [Something] colliding with [something] and b

In [24]:
avg_prob_in_diff_cases(df, session_type='magnet_2', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet_2 =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
    session  videos  prob_mean  prob_std  top1acc                                     examples
0  magnet_2      15     0.0112    0.0287   0.6667  260930-141745, 260930-141807, 260930-141829
==================== Videos ====================
         video_id   cam   session          placeholders                                 template label top1_pred correct_top1 correct_top5 prob_107 prob_73 prob_11 prob_131 prob_19
91  260930-141745  cam1  magnet_2  magnet, water bottle  Putting [something] next to [something]   107       107         True         True   0.9347  0.0171  0.0134   0.0122  0.0110

Top 5 Probability Breakdown:
  prob_107 ->  0.93469  |  Putting [something] next to [something]
  prob_73  ->  0.01711  |  Pretending to put [something] next to [something]
  prob_11  ->  0.01337  |  Dropping [something] next to [something]
  prob_131 ->  0.01215  |  [So

### "common objects in special actions: ":

- plugging_1 shows 3 videos of special action "Plugging [something] into [something] " with common objects (not a plug)
- To verify if everything is working as intended for these special actions.

- I have plugging_1, showing_1, attaching_1, piling_1 for "common objects in special actions" in recording session 2

In [26]:
avg_prob_in_diff_cases(df, session_type='plugging', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plugging =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session  videos  prob_mean  prob_std  top1acc                                     examples
0  plugging_1       3     0.8139    0.0658   1.0000  260930-144612, 260930-144637, 260930-144729
==================== Videos ====================
          video_id   cam     session  placeholders                               template label top1_pred correct_top1 correct_top5 prob_49 prob_50 prob_1 prob_161 prob_19
181  260930-144612  cam1  plugging_1  cable, phone  Plugging [something] into [something]    49        49         True         True  0.8353  0.0878 0.0505   0.0097  0.0036

Top 5 Probability Breakdown:
  prob_49  ->  0.83529  |  Plugging [something] into [something]
  prob_50  ->  0.08784  |  Plugging [something] into [something] but pulling it right out as you remove your hand
  prob_1   ->  0.05052  |  Attaching [something] to [something]
  prob_161 

In [27]:
avg_prob_in_diff_cases(df, session_type='showing', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= showing =========================
label 124: Showing a photo of [something] to the camera
cams used: ['cam1']
              session  videos  prob_mean  prob_std  top1acc                                     examples
0  showing_photo_of_1       3     0.6520    0.5103   0.6667  260930-145029, 260930-145050, 260930-145114
==================== Videos ====================
          video_id   cam             session placeholders                                      template label top1_pred correct_top1 correct_top5 prob_124 prob_0 prob_128 prob_168 prob_44
190  260930-145029  cam1  showing_photo_of_1     building  Showing a photo of [something] to the camera   124       124         True         True   0.9841 0.0079   0.0062   0.0003  0.0002

Top 5 Probability Breakdown:
  prob_124 ->  0.98411  |  Showing a photo of [something] to the camera
  prob_0   ->  0.00791  |  Approaching [something] with your camera
  prob_128 ->  0.00621  |  Showing [something] to the camer

In [28]:
avg_prob_in_diff_cases(df, session_type='piling', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= piling =========================
label 48: Piling [something] up
cams used: ['cam1']
    session  videos  prob_mean  prob_std  top1acc                                     examples
0  piling_1       3     0.1426    0.1562   0.0000  260930-150113, 260930-150230, 260930-150433
==================== Videos ====================
          video_id   cam   session placeholders               template label top1_pred correct_top1 correct_top5 prob_144 prob_48 prob_114 prob_102 prob_109
208  260930-150113  cam1  piling_1       plates  Piling [something] up    48       144        False         True   0.8684  0.1032   0.0132   0.0056   0.0034

Top 5 Probability Breakdown:
  prob_144 ->  0.86838  |  Stacking [number of] [something]
  prob_48  ->  0.10319  |  Piling [something] up
  prob_114 ->  0.01316  |  Putting [something similar to other things that are already on the table]
  prob_102 ->  0.00564  |  Putting [number of] [something] onto [something]
  prob_109 ->  0.003

In [29]:
avg_prob_in_diff_cases(df, session_type='attaching', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= attaching =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
       session  videos  prob_mean  prob_std  top1acc                                     examples
0  attaching_1       3     0.1768    0.2196   0.3333  260930-145307, 260930-145329, 260930-145941
==================== Videos ====================
          video_id   cam      session  placeholders                              template label top1_pred correct_top1 correct_top5 prob_162 prob_66 prob_3 prob_1 prob_161
199  260930-145307  cam1  attaching_1  pen cap, pen  Attaching [something] to [something]     1       162        False         True   0.2300  0.1045 0.0923 0.0785   0.0713

Top 5 Probability Breakdown:
  prob_162 ->  0.23004  |  Trying to bend [something unbendable] so nothing happens
  prob_66  ->  0.10452  |  Pretending to close [something] without actually closing it
  prob_3   ->  0.09230  |  Bending [something] until it breaks
  prob_1   ->  0.07